# Generate one sample per EMNIST class

Standalone notebook: loads the trained VAE checkpoint (`vae_best.ckpt`) and the
trained Latent Denoiser checkpoint (`latent_denoiser_best.ckpt`) and generates
one conditioned sample for each of the 62 EMNIST classes (0-9, A-Z, a-z), using
the deterministic (DDIM-style) reverse diffusion sampler.

It does not retrain anything — only inference from the saved checkpoints.

> Run on: Runtime -> Change Runtime Type -> T4 GPU (or CPU, just slower)

## 1. Imports and configuration

In [ ]:
import math
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

VAE_CKPT = 'training_checkpoints/vae_best.ckpt'
DENOISER_CKPT = 'training_checkpoints/latent_denoiser_best.ckpt'
OUTPUT_PATH = 'results/generated_samples_per_class.png'

## 2. VAE architecture (copied from VAE_EMNIST.ipynb)

These classes must match exactly what was used to train the VAE, otherwise
`load_state_dict` will fail. Both 2-block and 3-block variants are defined;
the checkpoint stores which architecture was actually used.

In [ ]:
class ReshapeToImage(nn.Module):
    def forward(self, x):
        return x.view(-1, 1, 28, 28)


# ── 2-block architecture ───────────────────────────────────────────

class Encoder2(nn.Module):
    def __init__(self, latent_dim, use_diff_sigma_E):
        super().__init__()
        self.conv = nn.Sequential(
            ReshapeToImage(),
            nn.Conv2d(1,  32, kernel_size=4, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=4, stride=2, padding=1),
            nn.ReLU(),
            nn.Flatten(),
        )
        self.fc_mu     = nn.Linear(64 * 7 * 7, latent_dim)
        logvar_out     = latent_dim if use_diff_sigma_E else 1
        self.fc_logvar = nn.Linear(64 * 7 * 7, logvar_out)

    def forward(self, x):
        h = self.conv(x)
        return self.fc_mu(h), self.fc_logvar(h)


class Decoder2(nn.Module):
    def __init__(self, latent_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(latent_dim, 64 * 7 * 7),
            nn.ReLU(),
            nn.Unflatten(1, (64, 7, 7)),
            nn.ConvTranspose2d(64, 32, kernel_size=4, stride=2, padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(32,  1, kernel_size=4, stride=2, padding=1),
            nn.Sigmoid(),
            nn.Flatten(),
        )

    def forward(self, z):
        return self.net(z)


# ── 3-block architecture ───────────────────────────────────────────

class Encoder3(nn.Module):
    def __init__(self, latent_dim, use_diff_sigma_E):
        super().__init__()
        self.conv = nn.Sequential(
            ReshapeToImage(),
            nn.Conv2d(1,   32, kernel_size=4, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv2d(32,  64, kernel_size=4, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1),
            nn.ReLU(),
            nn.Flatten(),
        )
        self.fc_mu     = nn.Linear(128 * 3 * 3, latent_dim)
        logvar_out     = latent_dim if use_diff_sigma_E else 1
        self.fc_logvar = nn.Linear(128 * 3 * 3, logvar_out)

    def forward(self, x):
        h = self.conv(x)
        return self.fc_mu(h), self.fc_logvar(h)


class Decoder3(nn.Module):
    def __init__(self, latent_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(latent_dim, 128 * 3 * 3),
            nn.ReLU(),
            nn.Unflatten(1, (128, 3, 3)),
            nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2,
                               padding=1, output_padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(64,  32, kernel_size=4, stride=2, padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(32,   1, kernel_size=4, stride=2, padding=1),
            nn.Sigmoid(),
            nn.Flatten(),
        )

    def forward(self, z):
        return self.net(z)


_ARCHITECTURES = {
    '2blocks': (Encoder2, Decoder2),
    '3blocks': (Encoder3, Decoder3),
}

def get_encoder_decoder(arch, latent_dim, use_diff_sigma_E):
    EncoderCls, DecoderCls = _ARCHITECTURES[arch]
    return EncoderCls(latent_dim, use_diff_sigma_E), DecoderCls(latent_dim)

## 3. Load the VAE checkpoint (encoder + decoder)

In [ ]:
vae_ckpt = torch.load(VAE_CKPT, map_location=device)

LATENT_DIM       = vae_ckpt['latent_dim']
USE_DIFF_SIGMA_E = vae_ckpt['use_diff_sigma_E']
ARCH             = vae_ckpt.get('arch', '2blocks')   # backward compat

_, decoder = get_encoder_decoder(ARCH, LATENT_DIM, USE_DIFF_SIGMA_E)
decoder = decoder.to(device)
decoder.load_state_dict(vae_ckpt['decoder'])
decoder.eval()
for p in decoder.parameters():
    p.requires_grad_(False)

print(f'Loaded VAE checkpoint from epoch {vae_ckpt["epoch"]} '
      f'(test loss {vae_ckpt["test_loss"]:.4f})')
print(f'LATENT_DIM = {LATENT_DIM}, ARCH = {ARCH}')

## 4. Latent Denoiser architecture (copied from Latent_Diffusion_EMNIST.ipynb)

In [ ]:
class LatentDenoiser(nn.Module):
    def __init__(self, latent_dim, num_classes, time_embedding_dim=256, hidden_dim=256, label_embedding_dim=64):
        super().__init__()
        self.latent_dim = latent_dim
        self.time_embedding_dim = time_embedding_dim
        self.hidden_dim = hidden_dim
        self.num_classes = num_classes
        self.label_embedding_dim = label_embedding_dim

        self.time_mlp = nn.Sequential(
            nn.Linear(time_embedding_dim, time_embedding_dim * 4),
            nn.GELU(),
            nn.Linear(time_embedding_dim * 4, time_embedding_dim)
        )

        self.label_emb = nn.Embedding(num_classes, label_embedding_dim)

        self.proj_in = nn.Linear(latent_dim + time_embedding_dim + label_embedding_dim, hidden_dim)

        self.down1 = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim * 2),
            nn.GELU(),
            nn.Linear(hidden_dim * 2, hidden_dim * 2)
        )
        self.down_res1 = nn.Linear(hidden_dim, hidden_dim * 2)

        self.down2 = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim * 4),
            nn.GELU(),
            nn.Linear(hidden_dim * 4, hidden_dim * 4)
        )
        self.down_res2 = nn.Linear(hidden_dim * 2, hidden_dim * 4)

        self.mid = nn.Sequential(
            nn.Linear(hidden_dim * 4, hidden_dim * 4),
            nn.GELU(),
            nn.Linear(hidden_dim * 4, hidden_dim * 4)
        )

        self.up1 = nn.Sequential(
            nn.Linear(hidden_dim * 4 + hidden_dim * 4, hidden_dim * 2),
            nn.GELU(),
            nn.Linear(hidden_dim * 2, hidden_dim * 2)
        )
        self.up_res1 = nn.Linear(hidden_dim * 4 + hidden_dim * 4, hidden_dim * 2)

        self.up2 = nn.Sequential(
            nn.Linear(hidden_dim * 2 + hidden_dim * 2, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim)
        )
        self.up_res2 = nn.Linear(hidden_dim * 2 + hidden_dim * 2, hidden_dim)

        self.proj_out = nn.Linear(hidden_dim, latent_dim)

    def gelu(self, x):
        return nn.functional.gelu(x)

    def forward(self, z, t, labels):
        t_embed = self.time_embedding(t)
        t_embed = self.time_mlp(t_embed)
        label_embed = self.label_emb(labels)

        x = torch.cat([z, t_embed, label_embed], dim=1)
        x = self.proj_in(x)

        h1 = x
        x = self.down1(x) + self.down_res1(h1)
        x = self.gelu(x)
        skip1 = x

        h2 = x
        x = self.down2(x) + self.down_res2(h2)
        x = self.gelu(x)
        skip2 = x

        x = self.mid(x) + x
        x = self.gelu(x)

        x_up1_input = torch.cat([x, skip2], dim=1)
        h3 = x_up1_input
        x = self.up1(x_up1_input) + self.up_res1(h3)
        x = self.gelu(x)

        x_up2_input = torch.cat([x, skip1], dim=1)
        h4 = x_up2_input
        x = self.up2(x_up2_input) + self.up_res2(h4)
        x = self.gelu(x)

        x = self.proj_out(x)
        return x

    def time_embedding(self, t):
        half_dim = self.time_embedding_dim // 2
        emb = torch.log(torch.tensor(10000.0)) / (half_dim - 1)
        emb = torch.exp(torch.arange(half_dim, device=t.device) * -emb)
        emb = t[:, None] * emb[None, :]
        emb = torch.cat((emb.sin(), emb.cos()), dim=-1)
        return emb

## 5. Load the Latent Denoiser checkpoint

In [ ]:
denoiser_ckpt = torch.load(DENOISER_CKPT, map_location=device)

NUM_EMNIST_CLASSES = denoiser_ckpt['num_emnist_classes']
TIMESTEPS          = denoiser_ckpt['timesteps']

latent_denoiser = LatentDenoiser(
    latent_dim=denoiser_ckpt['latent_dim'],
    num_classes=NUM_EMNIST_CLASSES,
    time_embedding_dim=denoiser_ckpt['time_embedding_dim'],
    hidden_dim=denoiser_ckpt['hidden_dim'],
    label_embedding_dim=denoiser_ckpt['label_embedding_dim'],
).to(device)
latent_denoiser.load_state_dict(denoiser_ckpt['model_state_dict'])
latent_denoiser.eval()
for p in latent_denoiser.parameters():
    p.requires_grad_(False)

print(f'Loaded Latent Denoiser checkpoint from epoch {denoiser_ckpt["epoch"]} '
      f'(final loss {denoiser_ckpt["final_loss"]:.4f})')
print(f'LATENT_DIM = {denoiser_ckpt["latent_dim"]}, NUM_EMNIST_CLASSES = {NUM_EMNIST_CLASSES}, '
      f'TIMESTEPS = {TIMESTEPS}')

assert denoiser_ckpt['latent_dim'] == LATENT_DIM, (
    'VAE and denoiser checkpoints disagree on latent_dim - make sure both files '
    'come from the same training run.'
)

## 6. Noise schedule and deterministic (DDIM-style) sampler

Same linear beta schedule used during training, and the deterministic reverse
step (no injected noise, eta=0) required by the assignment: `z[k]` is obtained
from `z[k+1]` and the predicted `x_0`, unlike the stochastic DDPM ancestral
sampler.

In [ ]:
def linear_beta_schedule(timesteps):
    scale = 1000 / timesteps
    beta_start = scale * 0.0001
    beta_end = scale * 0.02
    return torch.linspace(beta_start, beta_end, timesteps, dtype=torch.float32)

b_t = linear_beta_schedule(TIMESTEPS).to(device)
a_t = 1. - b_t
alpha_bar_t = torch.cumprod(a_t, dim=0)

def extract(a, t, x_shape):
    batch_size = t.shape[0]
    out = a.gather(-1, t).to(t.device)
    return out.view(batch_size, 1)

@torch.no_grad()
def p_sample(model, x, t, t_index, labels):
    # Deterministic reverse diffusion step (DDIM, eta=0)
    sqrt_alphas_cumprod_t = extract(torch.sqrt(alpha_bar_t), t, x.shape)
    sqrt_one_minus_alphas_cumprod_t = extract(torch.sqrt(1. - alpha_bar_t), t, x.shape)

    pred_noise = model(x, t, labels)

    x_0_pred = (x - sqrt_one_minus_alphas_cumprod_t * pred_noise) / sqrt_alphas_cumprod_t

    if t_index == 0:
        return x_0_pred

    t_prev = t - 1
    alpha_bar_t_prev = extract(alpha_bar_t, t_prev, x.shape)
    x_prev = torch.sqrt(alpha_bar_t_prev) * x_0_pred + torch.sqrt(1. - alpha_bar_t_prev) * pred_noise

    return x_prev

@torch.no_grad()
def p_sample_loop(model, shape, timesteps, device, labels):
    img = torch.randn(shape, device=device)
    for i in reversed(range(0, timesteps)):
        t = torch.full((shape[0],), i, device=device, dtype=torch.long)
        img = p_sample(model, img, t, i, labels)
    return img

@torch.no_grad()
def generate_samples(model, vae_decoder, num_samples, latent_dim, timesteps, device, conditioning_labels):
    model.eval()
    vae_decoder.eval()
    conditioning_labels = conditioning_labels.to(device).long()
    sampled_latents = p_sample_loop(model, (num_samples, latent_dim), timesteps, device, conditioning_labels)
    generated_images = vae_decoder(sampled_latents)
    return generated_images

## 7. Generate one sample per EMNIST class

In [ ]:
def fix_emnist_image(img: np.ndarray) -> np.ndarray:
    """Undo EMNIST's built-in 90-degree CW rotation + horizontal mirror (visualization only)."""
    return np.rot90(np.fliplr(img))

def get_emnist_char(class_id):
    if 0 <= class_id <= 9:
        return str(class_id)
    elif 10 <= class_id <= 35:
        return chr(ord('A') + class_id - 10)
    elif 36 <= class_id <= 61:
        return chr(ord('a') + class_id - 36)
    return '?'

conditioning_labels = torch.arange(NUM_EMNIST_CLASSES, device=device, dtype=torch.long)

generated_images = generate_samples(
    latent_denoiser, decoder, NUM_EMNIST_CLASSES, LATENT_DIM, TIMESTEPS, device, conditioning_labels
)

n_cols = 9
n_rows = (NUM_EMNIST_CLASSES + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 2, n_rows * 2))
axes = axes.flatten()

for i in range(NUM_EMNIST_CLASSES):
    img_raw = generated_images[i].cpu().view(28, 28).numpy()
    img_fixed = fix_emnist_image(img_raw)
    ax = axes[i]
    ax.imshow(img_fixed, cmap='gray')
    ax.axis('off')
    ax.set_title(f'{get_emnist_char(conditioning_labels[i].item())}', fontsize=10)

for j in range(NUM_EMNIST_CLASSES, len(axes)):
    axes[j].axis('off')

plt.suptitle('One generated sample per EMNIST class (deterministic latent diffusion)', fontsize=16)
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.savefig(OUTPUT_PATH, dpi=150)
plt.show()

print(f'Generated {NUM_EMNIST_CLASSES} samples (one per class), saved to {OUTPUT_PATH}')